# Sentiment Analysis using NLP Pipeline & Machine Learning

## Objective
The goal of this project is to build a Sentiment Analysis system that classifies text reviews into positive or negative sentiment.

The project follows a complete Natural Language Processing (NLP) pipeline including:
- Data Understanding
- Text Preprocessing
- Feature Engineering
- Model Training
- Model Evaluation
- Performance Comparison

Machine Learning models such as Logistic Regression, Naive Bayes, and Decision Tree will be trained and compared using evaluation metrics like Accuracy, Precision, Recall, and F1 Score.

## Importing Required Libraries

In [2]:
import pandas as pd
import numpy as np
import re
import string

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Feature Engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Model evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Train test split
from sklearn.model_selection import train_test_split

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...


True

## Data Understanding

In [3]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [6]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
print(df['review'][0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

## NLP Preprocessing

In [8]:
#Convert text to lowercase
df['review'] = df['review'].str.lower()

df['review'][0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />i would say the main appeal of the show is due to the fa

In [9]:
#Removing Punctuation 
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['review'] = df['review'].apply(remove_punctuation)

In [11]:
#Tokenization
df['tokens'] = df['review'].apply(word_tokenize)

df['tokens'][0]

['one',
 'of',
 'the',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 '1',
 'oz',
 'episode',
 'youll',
 'be',
 'hooked',
 'they',
 'are',
 'right',
 'as',
 'this',
 'is',
 'exactly',
 'what',
 'happened',
 'with',
 'mebr',
 'br',
 'the',
 'first',
 'thing',
 'that',
 'struck',
 'me',
 'about',
 'oz',
 'was',
 'its',
 'brutality',
 'and',
 'unflinching',
 'scenes',
 'of',
 'violence',
 'which',
 'set',
 'in',
 'right',
 'from',
 'the',
 'word',
 'go',
 'trust',
 'me',
 'this',
 'is',
 'not',
 'a',
 'show',
 'for',
 'the',
 'faint',
 'hearted',
 'or',
 'timid',
 'this',
 'show',
 'pulls',
 'no',
 'punches',
 'with',
 'regards',
 'to',
 'drugs',
 'sex',
 'or',
 'violence',
 'its',
 'is',
 'hardcore',
 'in',
 'the',
 'classic',
 'use',
 'of',
 'the',
 'wordbr',
 'br',
 'it',
 'is',
 'called',
 'oz',
 'as',
 'that',
 'is',
 'the',
 'nickname',
 'given',
 'to',
 'the',
 'oswald',
 'maximum',
 'security',
 'state',
 'penitentary',
 'it',
 'focuses',
 

In [14]:
#Removing Stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens'] = df['tokens'].apply(remove_stopwords)

In [15]:
#Lemmatization
lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['tokens'] = df['tokens'].apply(lemmatize_words)

In [16]:
#Removing URLs
def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text)

df['review'] = df['review'].apply(remove_urls)

In [17]:
#Joining Tokens
df['clean_review'] = df['tokens'].apply(lambda x: " ".join(x))

df[['review','clean_review']].head()

,review,clean_review
0,one of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode y...
1,a wonderful little production br br the filmin...,wonderful little production br br filming tech...
2,i thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,basically theres a family where a little boy j...,basically family little boy jake think zombie ...
4,petter matteis love in the time of money is a ...,petter matteis love time money visually stunni...


## Task3

### Feature Engineering

In [18]:
#Train- Test Data Split
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (40000,)
Testing samples: (10000,)


In [19]:
#Bag of Words(BOW)
bow_vectorizer = CountVectorizer(max_features=5000)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [20]:
X_train_bow.shape

(40000, 5000)

In [21]:
#TF- IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [22]:
X_train_tfidf.shape

(40000, 5000)

# Task 4

# Model Building

In [23]:
#Model Evaluation Function
def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Train model
    model.fit(X_train, y_train)

    # Predict
    predictions = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, pos_label='positive')
    recall = recall_score(y_test, predictions, pos_label='positive')
    f1 = f1_score(y_test, predictions, pos_label='positive')

    return accuracy, precision, recall, f1

In [24]:
# Logistic 
lr = LogisticRegression(max_iter=1000)
lr_bow = evaluate_model(lr, X_train_bow, X_test_bow, y_train, y_test)
lr_tfidf = evaluate_model(lr, X_train_tfidf, X_test_tfidf, y_train, y_test)

In [25]:
# Naive Bayes
nb = MultinomialNB()
nb_bow = evaluate_model(nb, X_train_bow, X_test_bow, y_train, y_test)
nb_tfidf = evaluate_model(nb, X_train_tfidf, X_test_tfidf, y_train, y_test)

In [26]:
# Decision Tree
dt = DecisionTreeClassifier()
dt_bow = evaluate_model(dt, X_train_bow, X_test_bow, y_train, y_test)
dt_tfidf = evaluate_model(dt, X_train_tfidf, X_test_tfidf, y_train, y_test)

# Task5

# Model Evaluation

In [27]:
# Model Performance Comparison
results = pd.DataFrame({

    "Model": [
        "Logistic Regression (BoW)",
        "Logistic Regression (TF-IDF)",
        "Naive Bayes (BoW)",
        "Naive Bayes (TF-IDF)",
        "Decision Tree (BoW)",
        "Decision Tree (TF-IDF)"
    ],

    "Accuracy": [
        lr_bow[0], lr_tfidf[0],
        nb_bow[0], nb_tfidf[0],
        dt_bow[0], dt_tfidf[0]
    ],

    "Precision": [
        lr_bow[1], lr_tfidf[1],
        nb_bow[1], nb_tfidf[1],
        dt_bow[1], dt_tfidf[1]
    ],

    "Recall": [
        lr_bow[2], lr_tfidf[2],
        nb_bow[2], nb_tfidf[2],
        dt_bow[2], dt_tfidf[2]
    ],

    "F1 Score": [
        lr_bow[3], lr_tfidf[3],
        nb_bow[3], nb_tfidf[3],
        dt_bow[3], dt_tfidf[3]
    ]

})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression (BoW),0.8766,0.872966,0.883707,0.878304
1,Logistic Regression (TF-IDF),0.8882,0.878841,0.902560,0.890542
2,Naive Bayes (BoW),0.8484,0.852794,0.845009,0.848884
3,Naive Bayes (TF-IDF),0.8537,0.852663,0.857908,0.855277
4,Decision Tree (BoW),0.7178,0.722635,0.714031,0.718307
5,Decision Tree (TF-IDF),0.7241,0.728457,0.721373,0.724898


# Task6

# Comparison & Insights

# Best Preprocessing Steps
## Best Preprocessing Steps

Several preprocessing techniques were applied to clean and standardize the text data. Among these steps, the most impactful ones were:

**1. Stopword Removal**

Stopwords such as "the", "is", "and", and "in" appear very frequently in text but do not contribute to sentiment classification. Removing these words helps reduce noise and allows the model to focus on meaningful words such as "amazing", "boring", or "terrible" that carry sentiment.

**2. Lemmatization**

Lemmatization converts words into their base forms. For example, "running", "runs", and "ran" become "run". This helps group similar words together and reduces the size of the vocabulary, making it easier for the model to learn patterns.

**3. Removing Punctuation and Special Characters**

Punctuation marks and special characters do not add meaningful information for sentiment classification. Removing them simplifies the text and ensures cleaner tokenization.

**4. Lowercasing**

Lowercasing ensures that words like "Movie" and "movie" are treated as the same word, which prevents duplication in the vocabulary and improves consistency.

Overall, stopword removal and lemmatization had the greatest impact because they reduced unnecessary words and normalized the vocabulary, allowing the machine learning models to focus on sentiment-carrying terms.

# Best Vectorization Method
Two vectorization methods were used to convert text into numerical features:

• Bag of Words (BoW)  
• TF-IDF

From the results, TF-IDF performed slightly better across most models.


## Best Performing Model

Among the tested models, Logistic Regression with TF-IDF achieved the highest accuracy and F1 score.

Accuracy: 0.8882  
F1 Score: 0.8905

Logistic Regression works well for text classification because it handles high-dimensional data effectively and learns relationships between features and target labels.

## Model Trade-offs
```
Logistic Regression
• Highest accuracy among all models
• Works well with high dimensional text data
• Slightly slower to train than Naive Bayes

Naive Bayes
• Very fast and efficient
• Performs well for text classification
• Slightly lower accuracy than Logistic Regression

Decision Tree
• Easy to interpret
• Lower accuracy due to overfitting on high dimensional text features
```

In [28]:
#Testing the model on new text

sample_review = ["This movie was absolutely fantastic and I loved every moment"]

# Preprocess
sample_review = [remove_urls(text) for text in sample_review]
sample_review = [remove_punctuation(text) for text in sample_review]
sample_review = [text.lower() for text in sample_review]

sample_review = [word_tokenize(text) for text in sample_review]
sample_review = [[word for word in tokens if word not in stop_words] for tokens in sample_review]
sample_review = [[lemmatizer.lemmatize(word) for word in tokens] for tokens in sample_review]

sample_review = [" ".join(tokens) for tokens in sample_review]

# Vectorize using TF-IDF
sample_vector = tfidf_vectorizer.transform(sample_review)

prediction = lr.predict(sample_vector)

print("Predicted Sentiment:", prediction[0])

Predicted Sentiment: positive
